# Plant Disease Recognition: MobileNetV2 (the model the app uses)

This notebook trains and evaluates the **MobileNetV2 transfer-learning model** that `main.py` loads from
`trained_plant_disease_model.keras`. The older `Train_plant_disease.ipynb` trains a different model
(a custom CNN from scratch) and is kept only as a baseline.

**What this notebook fixes compared to the old one**

| Problem in the old notebook | Fix here |
| --- | --- |
| Training code for the shipped MobileNetV2 model was missing | Full architecture + training below |
| `valid/` was used both to tune training *and* to report the final score | `train/` is split 90/10 into train/validation; `valid/` is kept as an untouched **test set** |
| Validation loss rose after epoch ~5 (overfitting), but training kept going | `EarlyStopping` restores the best epoch |
| Reported accuracy belonged to a model the app does not use | Section 3 evaluates the **shipped** `.keras` file directly |

Dataset: [New Plant Diseases Dataset (Kaggle)](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset).
Note: that dataset was built with offline augmentation, so near-duplicate images of the same leaf can sit in
both `train/` and `valid/`. Treat the test score as optimistic for real-world photos.

## 1. Setup

In [ ]:
import os, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow", tf.__version__)

In [ ]:
# ---- Configuration ----
DATA_DIR   = None            # folder that contains `train/` and `valid/`; None = download with kagglehub
IMG_SIZE   = (128, 128)      # must match main.py
BATCH_SIZE = 32
SEED       = 42
VAL_SPLIT  = 0.10            # share of train/ held out for validation (early stopping)
EPOCHS     = 15              # upper limit; EarlyStopping usually stops sooner
FINE_TUNE  = False           # True = second phase that unfreezes the top of MobileNetV2
FINE_TUNE_EPOCHS = 5
WEIGHTS    = "imagenet"      # pre-trained backbone weights
SAVE_PATH  = "mobilenetv2_plant_disease.keras"   # copy over trained_plant_disease_model.keras to ship it
SHIPPED_MODEL = "trained_plant_disease_model.keras"

In [ ]:
def find_split_dirs(root):
    """Return (train_dir, test_dir), searching nested folders (the Kaggle zip nests them)."""
    for dirpath, dirnames, _ in os.walk(root):
        if "train" in dirnames and "valid" in dirnames:
            return os.path.join(dirpath, "train"), os.path.join(dirpath, "valid")
    raise FileNotFoundError(f"No folder with both train/ and valid/ under {root}")

if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

TRAIN_DIR, TEST_DIR = find_split_dirs(DATA_DIR)
print("train:", TRAIN_DIR)
print("test :", TEST_DIR)

## 2. Data: train / validation / test

- **Train** (90% of `train/`): the model learns from these.
- **Validation** (10% of `train/`): checked after each epoch to decide when to stop.
- **Test** (`valid/`): used **once**, at the end. Never influences training.

In [ ]:
common = dict(labels="inferred", label_mode="categorical", color_mode="rgb",
              batch_size=BATCH_SIZE, image_size=IMG_SIZE, interpolation="bilinear")

train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="both", seed=SEED, shuffle=True, **common)

# shuffle=False keeps a fixed order, so predictions line up with labels for the report
test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, shuffle=False, **common)

class_names = train_ds.class_names
assert class_names == test_ds.class_names, "train/ and valid/ have different class folders"
print(len(class_names), "classes")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
# The app's CLASS_NAMES list must be in exactly this order (Keras sorts folders alphabetically).
# This check catches a mismatch that would silently mislabel every prediction.
import ast, pathlib
src = pathlib.Path("main.py").read_text(encoding="utf-8") if pathlib.Path("main.py").exists() else None
if src:
    app_names = next(ast.literal_eval(n.value) for n in ast.parse(src).body
                     if isinstance(n, ast.Assign) and getattr(n.targets[0], "id", "") == "CLASS_NAMES")
    assert app_names == class_names, "main.py CLASS_NAMES order differs from the dataset folders"
    print("main.py CLASS_NAMES matches the dataset order")

In [ ]:
def evaluate(model, ds, title):
    """Accuracy, per-class report and confusion matrix on a dataset."""
    y_prob = model.predict(ds, verbose=0)
    y_pred = y_prob.argmax(axis=1)
    y_true = np.concatenate([y.numpy().argmax(axis=1) for _, y in ds])
    acc = float((y_pred == y_true).mean())
    print(f"{title}: accuracy = {acc:.4f} on {len(y_true)} images")
    print(classification_report(y_true, y_pred, labels=range(len(class_names)),
                                target_names=class_names, digits=3, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    plt.figure(figsize=(18, 15))
    sns.heatmap(cm, cmap="Greens", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted class"); plt.ylabel("Actual class"); plt.title(title)
    plt.tight_layout(); plt.show()
    report = classification_report(y_true, y_pred, labels=range(len(class_names)),
                                   target_names=class_names, output_dict=True, zero_division=0)
    return {"accuracy": acc, "n_images": int(len(y_true)), "report": report}

## 3. Evaluate the shipped model (no training needed)

Run this to get an honest test score for the model the app uses today. You can stop here if you only need
the numbers. Note: if the shipped model was trained on all of `train/` and `valid/` was used for its early
stopping, this score is slightly optimistic.

In [ ]:
if os.path.exists(SHIPPED_MODEL):
    shipped = tf.keras.models.load_model(SHIPPED_MODEL)
    shipped_metrics = evaluate(shipped, test_ds, "Shipped model on test set (valid/)")
    with open("metrics_shipped_model.json", "w") as f:
        json.dump(shipped_metrics, f, indent=2)
else:
    print("Shipped model not found next to this notebook; skipping.")

## 4. Build the model

Same architecture as the shipped file: augmentation → rescale to [-1, 1] → frozen MobileNetV2 →
global average pooling → Dense 512 head → 38-way softmax. Augmentation layers only act during training,
and rescaling is inside the model, so `main.py` can pass raw 0–255 pixels.

In [ ]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.1, fill_mode="reflect"),
    layers.RandomContrast(0.1),
], name="data_augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights=WEIGHTS)
base_model.trainable = False   # phase 1: only the head learns

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="rescale_to_minus1_1")(x)
x = base_model(x, training=False)   # keep BatchNorm in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs, name="plant_disease_mobilenetv2")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train (with early stopping)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}

In [ ]:
if FINE_TUNE:
    # Phase 2: unfreeze the top ~30 layers of MobileNetV2 and train slowly.
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    ft = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=callbacks[:1])
    for k, vals in ft.history.items():
        hist.setdefault(k, []).extend(float(v) for v in vals)

with open("training_hist_mobilenetv2.json", "w") as f:
    json.dump(hist, f, indent=2)

In [ ]:
ep = range(1, len(hist["accuracy"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, hist["accuracy"], label="train"); ax[0].plot(ep, hist["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(ep, hist["loss"], label="train"); ax[1].plot(ep, hist["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Final test score (run once)

In [ ]:
test_metrics = evaluate(model, test_ds, "New MobileNetV2 on test set (valid/)")
with open("metrics_mobilenetv2.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

model.save(SAVE_PATH)
print("Saved", SAVE_PATH, "- copy it over", SHIPPED_MODEL, "to use it in the app.")

## 7. Sanity check: the saved file predicts like the app will

In [ ]:
reloaded = tf.keras.models.load_model(SAVE_PATH)
images, labels = next(iter(test_ds.unbatch().shuffle(2000, seed=SEED).batch(8)))   # 8 random test images
probs = reloaded.predict(images, verbose=0)   # raw 0-255 pixels, exactly like main.py
for p, y in zip(probs, labels.numpy()):
    print(f"true: {class_names[y.argmax()]:45s} predicted: {class_names[p.argmax()]:45s} {p.max()*100:5.1f}%")